# SEC Financial Dashboard Project

## Project Overview

This project builds a financial performance dataset using SEC Financial Statement Data Sets from 2020 to 2025.

The objective is to extract, clean, transform, and prepare real public company financial data for a Power BI dashboard.

The analysis focuses on five major U.S. companies:

- Apple
- Microsoft
- Amazon
- Tesla
- JPMorgan Chase

In [22]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [4]:
# ============================================================
# 1. PROJECT PATHS
# ============================================================

In [24]:
base_path = os.path.expanduser("/Users/ankitatripathy/Desktop/PowerBi/Raw Data")

print("Folder exists:", os.path.exists(base_path))

if os.path.exists(base_path):
    print("Folders found:")
    print(sorted(os.listdir(base_path))[:10])

Folder exists: True
Folders found:
['.DS_Store', '2020q1', '2020q2', '2020q3', '2020q4', '2021q1', '2021q2', '2021q3', '2021q4', '2022q1']


In [26]:
target_ciks = [
    320193,     # Apple
    789019,     # Microsoft
    1018724,    # Amazon
    1318605,    # Tesla
    19617       # JPMorgan Chase
]

company_map = {
    "APPLE INC": "Apple",
    "MICROSOFT CORP": "Microsoft",
    "AMAZON COM INC": "Amazon",
    "TESLA, INC.": "Tesla",
    "JPMORGAN CHASE & CO": "JPMorgan Chase"
}

sector_map = {
    "Amazon": "Technology",
    "Apple": "Technology",
    "Microsoft": "Technology",
    "Tesla": "Technology",
    "JPMorgan Chase": "Banking"
}

financial_tags = [
    "Revenues",
    "SalesRevenueNet",
    "RevenueFromContractWithCustomerExcludingAssessedTax",

    # JPMorgan banking revenue tags
    "RevenuesNetOfInterestExpense",
    "InterestIncomeExpenseNet",
    "NoninterestIncome",

    "GrossProfit",
    "OperatingIncomeLoss",
    "NetIncomeLoss",
    "Assets",
    "Liabilities",
    "StockholdersEquity",
    "CashAndCashEquivalentsAtCarryingValue",
    "NetCashProvidedByUsedInOperatingActivities"
]

In [28]:
master_list = []

quarter_folders = sorted([
    folder for folder in os.listdir(base_path)
    if os.path.isdir(os.path.join(base_path, folder))
])

print("Total folders found:", len(quarter_folders))

for folder in quarter_folders:
    print("Processing:", folder)

    sub_path = os.path.join(base_path, folder, "sub.txt")
    num_path = os.path.join(base_path, folder, "num.txt")

    if not os.path.exists(sub_path) or not os.path.exists(num_path):
        print("Missing sub.txt or num.txt in:", folder)
        continue

    sub = pd.read_csv(sub_path, sep="\t", low_memory=False)
    sub["cik"] = pd.to_numeric(sub["cik"], errors="coerce")

    sub_filtered = sub[
        (sub["cik"].isin(target_ciks)) &
        (sub["form"].isin(["10-Q", "10-K"]))
    ].copy()

    if sub_filtered.empty:
        continue

    needed_adsh = sub_filtered["adsh"].unique()

    num = pd.read_csv(num_path, sep="\t", low_memory=False)

    num_filtered = num[
        (num["adsh"].isin(needed_adsh)) &
        (num["tag"].isin(financial_tags)) &
        (num["uom"] == "USD") &
        (num["segments"].isna())
    ].copy()

    if num_filtered.empty:
        continue

    merged = pd.merge(
        num_filtered,
        sub_filtered[
            [
                "adsh",
                "name",
                "cik",
                "period",
                "filed",
                "form"
            ]
        ],
        on="adsh",
        how="inner"
    )

    merged["QuarterFolder"] = folder
    master_list.append(merged)

master = pd.concat(master_list, ignore_index=True)

print("Master shape:", master.shape)
master.head()

Total folders found: 24
Processing: 2020q1
Processing: 2020q2
Processing: 2020q3
Processing: 2020q4
Processing: 2021q1
Processing: 2021q2
Processing: 2021q3
Processing: 2021q4
Processing: 2022q1
Processing: 2022q2
Processing: 2022q3
Processing: 2022q4
Processing: 2023q1
Processing: 2023q2
Processing: 2023q3
Processing: 2023q4
Processing: 2024q1
Processing: 2024q2
Processing: 2024q3
Processing: 2024q4
Processing: 2025q1
Processing: 2025q2
Processing: 2025q3
Processing: 2025q4
Master shape: (3156, 16)


,adsh,tag,version,ddate,qtrs,uom,segments,coreg,value,footnote,name,cik,period,filed,form,QuarterFolder
0,0001018724-20-000004,RevenueFromContractWithCustomerExcludingAssess...,us-gaap/2019,20180930,1,USD,NaN,NaN,5.657600e+10,NaN,AMAZON COM INC,1018724,20191231.0,20200131,10-K,2020q1
1,0000019617-20-000257,NoninterestIncome,us-gaap/2019,20181231,4,USD,NaN,NaN,5.397000e+10,NaN,JPMORGAN CHASE & CO,19617,20191231.0,20200225,10-K,2020q1
2,0001564590-20-002450,CashAndCashEquivalentsAtCarryingValue,us-gaap/2019,20190630,0,USD,NaN,NaN,1.135600e+10,NaN,MICROSOFT CORP,789019,20191231.0,20200129,10-Q,2020q1
3,0001564590-20-004475,NetIncomeLoss,us-gaap/2019,20171231,4,USD,NaN,NaN,-1.962000e+09,NaN,"TESLA, INC.",1318605,20191231.0,20200213,10-K,2020q1
4,0001564590-20-004475,CashAndCashEquivalentsAtCarryingValue,us-gaap/2019,20191231,0,USD,NaN,NaN,6.268000e+09,NaN,"TESLA, INC.",1318605,20191231.0,20200213,10-K,2020q1


In [30]:
master["ddate"] = pd.to_datetime(
    master["ddate"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

master["filed"] = pd.to_datetime(
    master["filed"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

master["Year"] = master["ddate"].dt.year
master["Quarter"] = master["ddate"].dt.quarter

master = master[
    (master["Year"] >= 2020) &
    (master["Year"] <= 2025)
].copy()

print("Master after date filter:", master.shape)

master[
    [
        "name",
        "cik",
        "form",
        "ddate",
        "Year",
        "Quarter",
        "tag",
        "value"
    ]
].head()

Master after date filter: (2614, 18)


,name,cik,form,ddate,Year,Quarter,tag,value
169,JPMORGAN CHASE & CO,19617,10-Q,2020-03-31,2020,1,InterestIncomeExpenseNet,1.443900e+10
171,JPMORGAN CHASE & CO,19617,10-Q,2020-03-31,2020,1,NetCashProvidedByUsedInOperatingActivities,-1.207720e+11
172,AMAZON COM INC,1018724,10-Q,2020-03-31,2020,1,StockholdersEquity,6.527200e+10
177,"TESLA, INC.",1318605,10-Q,2020-03-31,2020,1,Liabilities,2.651800e+10
179,JPMORGAN CHASE & CO,19617,10-Q,2020-03-31,2020,1,Liabilities,2.878169e+12


In [32]:
financial_table = master.pivot_table(
    index=[
        "name",
        "cik",
        "form",
        "filed",
        "ddate",
        "Year",
        "Quarter"
    ],
    columns="tag",
    values="value",
    aggfunc="first"
).reset_index()

financial_table.columns.name = None

print("Financial table shape:", financial_table.shape)
print(financial_table.columns.tolist())

financial_table.head()

Financial table shape: (398, 20)
['name', 'cik', 'form', 'filed', 'ddate', 'Year', 'Quarter', 'Assets', 'CashAndCashEquivalentsAtCarryingValue', 'GrossProfit', 'InterestIncomeExpenseNet', 'Liabilities', 'NetCashProvidedByUsedInOperatingActivities', 'NetIncomeLoss', 'NoninterestIncome', 'OperatingIncomeLoss', 'RevenueFromContractWithCustomerExcludingAssessedTax', 'Revenues', 'RevenuesNetOfInterestExpense', 'StockholdersEquity']


,name,cik,form,filed,ddate,Year,Quarter,Assets,CashAndCashEquivalentsAtCarryingValue,GrossProfit,InterestIncomeExpenseNet,Liabilities,NetCashProvidedByUsedInOperatingActivities,NetIncomeLoss,NoninterestIncome,OperatingIncomeLoss,RevenueFromContractWithCustomerExcludingAssessedTax,Revenues,RevenuesNetOfInterestExpense,StockholdersEquity
0,AMAZON COM INC,1018724,10-K,2021-02-03,2020-03-31,2020,1,NaN,NaN,NaN,NaN,NaN,NaN,2.535000e+09,NaN,3.989000e+09,7.545200e+10,NaN,NaN,NaN
1,AMAZON COM INC,1018724,10-K,2021-02-03,2020-06-30,2020,2,NaN,NaN,NaN,NaN,NaN,NaN,5.243000e+09,NaN,5.843000e+09,8.891200e+10,NaN,NaN,NaN
2,AMAZON COM INC,1018724,10-K,2021-02-03,2020-09-30,2020,3,NaN,NaN,NaN,NaN,NaN,NaN,6.331000e+09,NaN,6.194000e+09,9.614500e+10,NaN,NaN,NaN
3,AMAZON COM INC,1018724,10-K,2021-02-03,2020-12-31,2020,4,3.211950e+11,4.212200e+10,NaN,NaN,NaN,6.606400e+10,7.222000e+09,NaN,2.289900e+10,1.255550e+11,NaN,NaN,9.340400e+10
4,AMAZON COM INC,1018724,10-K,2022-02-04,2020-12-31,2020,4,3.211950e+11,4.212200e+10,NaN,NaN,NaN,6.606400e+10,2.133100e+10,NaN,2.289900e+10,3.860640e+11,NaN,NaN,9.340400e+10


In [34]:
df = financial_table.copy()

df["Company"] = df["name"].replace(company_map)
df["Sector"] = df["Company"].map(sector_map)

df = df[df["Company"].isin(company_map.values())].copy()

df["Revenue"] = np.nan

revenue_tags = [
    "Revenues",
    "SalesRevenueNet",
    "RevenueFromContractWithCustomerExcludingAssessedTax",
    "RevenuesNetOfInterestExpense",
    "InterestIncomeExpenseNet",
    "NoninterestIncome"
]

for col in revenue_tags:
    if col in df.columns:
        df["Revenue"] = df["Revenue"].fillna(df[col])

df["Gross_Profit"] = df.get("GrossProfit")
df["Operating_Income"] = df.get("OperatingIncomeLoss")
df["Net_Income"] = df.get("NetIncomeLoss")
df["Operating_Cash_Flow"] = df.get("NetCashProvidedByUsedInOperatingActivities")
df["Equity"] = df.get("StockholdersEquity")

df[
    [
        "Company",
        "Sector",
        "Year",
        "Quarter",
        "Revenue",
        "Net_Income",
        "Operating_Cash_Flow",
        "Equity"
    ]
].head()

,Company,Sector,Year,Quarter,Revenue,Net_Income,Operating_Cash_Flow,Equity
0,Amazon,Technology,2020,1,7.545200e+10,2.535000e+09,NaN,NaN
1,Amazon,Technology,2020,2,8.891200e+10,5.243000e+09,NaN,NaN
2,Amazon,Technology,2020,3,9.614500e+10,6.331000e+09,NaN,NaN
3,Amazon,Technology,2020,4,1.255550e+11,7.222000e+09,6.606400e+10,9.340400e+10
4,Amazon,Technology,2020,4,3.860640e+11,2.133100e+10,6.606400e+10,9.340400e+10


In [36]:
def safe_divide(numerator, denominator):
    return np.where(
        (pd.notna(denominator)) & (denominator != 0),
        numerator / denominator,
        np.nan
    )

df["Profit_Margin"] = safe_divide(df["Net_Income"], df["Revenue"]) * 100
df["Gross_Margin"] = safe_divide(df["Gross_Profit"], df["Revenue"]) * 100
df["Operating_Margin"] = safe_divide(df["Operating_Income"], df["Revenue"]) * 100
df["ROE"] = safe_divide(df["Net_Income"], df["Equity"]) * 100
df["Cash_Flow_Margin"] = safe_divide(df["Operating_Cash_Flow"], df["Revenue"]) * 100

df[
    [
        "Company",
        "Year",
        "Quarter",
        "Profit_Margin",
        "ROE",
        "Cash_Flow_Margin"
    ]
].head()

,Company,Year,Quarter,Profit_Margin,ROE,Cash_Flow_Margin
0,Amazon,2020,1,3.359752,NaN,NaN
1,Amazon,2020,2,5.896842,NaN,NaN
2,Amazon,2020,3,6.584846,NaN,NaN
3,Amazon,2020,4,5.752061,7.732003,52.617578
4,Amazon,2020,4,5.525250,22.837352,17.112189


In [38]:
df["ROE_Adjusted"] = np.where(
    df["ROE"] > 100,
    df["ROE"] / 4,
    df["ROE"]
)

df["ROE_Flag"] = np.where(
    df["ROE"] > 100,
    "TTM_Anomaly_Adjusted",
    ""
)

df["OCF_Category"] = np.where(
    df["Company"] == "JPMorgan Chase",
    "Banking OCF",
    "Operating"
)

df["Cash_Flow_Margin_Adj"] = np.where(
    df["Company"] == "JPMorgan Chase",
    np.nan,
    df["Cash_Flow_Margin"]
)

df["JPMorgan_OCF_Note"] = np.where(
    df["Company"] == "JPMorgan Chase",
    "Bank OCF includes lending, deposit and securities activity; not directly comparable to technology firms.",
    ""
)

df["Has_Financial_Data"] = np.where(
    df[
        [
            "Revenue",
            "Net_Income",
            "Operating_Cash_Flow"
        ]
    ].notna().any(axis=1),
    1,
    0
)

df["Data_Period_Note"] = np.where(
    df["Has_Financial_Data"] == 1,
    "Q" + df["Quarter"].astype(str) + " " + df["Year"].astype(str) + " - Financial Data Available",
    "Q" + df["Quarter"].astype(str) + " " + df["Year"].astype(str) + " - Balance Sheet Only"
)

df["Profit_Margin_Clean"] = df["Profit_Margin"]

df[
    [
        "Company",
        "Year",
        "Quarter",
        "ROE",
        "ROE_Adjusted",
        "ROE_Flag",
        "Cash_Flow_Margin",
        "Cash_Flow_Margin_Adj",
        "Has_Financial_Data"
    ]
].head(10)

,Company,Year,Quarter,ROE,ROE_Adjusted,ROE_Flag,Cash_Flow_Margin,Cash_Flow_Margin_Adj,Has_Financial_Data
0,Amazon,2020,1,NaN,NaN,,NaN,NaN,1
1,Amazon,2020,2,NaN,NaN,,NaN,NaN,1
2,Amazon,2020,3,NaN,NaN,,NaN,NaN,1
3,Amazon,2020,4,7.732003,7.732003,,52.617578,52.617578,1
4,Amazon,2020,4,22.837352,22.837352,,17.112189,17.112189,1
5,Amazon,2021,4,24.133965,24.133965,,9.860543,9.860543,1
6,Amazon,2020,4,22.837352,22.837352,,17.112189,17.112189,1
7,Amazon,2021,4,24.133965,24.133965,,9.860543,9.860543,1
8,Amazon,2022,4,-1.863835,-1.863835,,9.096021,9.096021,1
9,Amazon,2020,4,NaN,NaN,,NaN,NaN,0


In [40]:
df = df.sort_values(
    [
        "Company",
        "Year",
        "Quarter"
    ]
).copy()

df["Revenue_YoY_Growth"] = (
    df.groupby(
        [
            "Company",
            "Quarter"
        ]
    )["Revenue"]
    .pct_change(periods=1)
    * 100
)

df[
    [
        "Company",
        "Year",
        "Quarter",
        "Revenue",
        "Revenue_YoY_Growth"
    ]
].head(20)

/var/folders/d2/3r95_bmx7ms7lt1g7c04lx2r0000gn/T/ipykernel_56485/3716624546.py:16: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change(periods=1)


,Company,Year,Quarter,Revenue,Revenue_YoY_Growth
0,Amazon,2020,1,7.545200e+10,NaN
17,Amazon,2020,1,7.545200e+10,0.000000
18,Amazon,2020,1,NaN,0.000000
22,Amazon,2020,1,7.545200e+10,0.000000
25,Amazon,2020,1,NaN,0.000000
1,Amazon,2020,2,8.891200e+10,NaN
19,Amazon,2020,2,8.891200e+10,0.000000
20,Amazon,2020,2,NaN,0.000000
26,Amazon,2020,2,8.891200e+10,0.000000
30,Amazon,2020,2,NaN,0.000000


In [42]:
def minmax_score(series):
    s = series.astype(float)
    mn = s.min(skipna=True)
    mx = s.max(skipna=True)

    if pd.isna(mn) or pd.isna(mx) or mx == mn:
        return pd.Series(np.nan, index=series.index)

    return ((s - mn) / (mx - mn)) * 100

df["ROE_for_FHS"] = df["ROE_Adjusted"]

df["FHS_PM_Score"] = (
    df.groupby("Year")["Profit_Margin_Clean"]
    .transform(minmax_score)
)

df["FHS_CFM_Score"] = (
    df.groupby("Year")["Cash_Flow_Margin_Adj"]
    .transform(minmax_score)
)

df["FHS_ROE_Score"] = (
    df.groupby("Year")["ROE_for_FHS"]
    .transform(minmax_score)
)

df["FHS_V2"] = (
    df["FHS_PM_Score"] * 0.35 +
    df["FHS_ROE_Score"] * 0.30 +
    df["FHS_CFM_Score"].fillna(50) * 0.35
)

df[
    [
        "Company",
        "Year",
        "Quarter",
        "FHS_PM_Score",
        "FHS_CFM_Score",
        "ROE_for_FHS",
        "FHS_V2"
    ]
].head(20)

,Company,Year,Quarter,FHS_PM_Score,FHS_CFM_Score,ROE_for_FHS,FHS_V2
0,Amazon,2020,1,2.662193,NaN,NaN,NaN
17,Amazon,2020,1,11.821827,6.838617,16.183049,12.007813
18,Amazon,2020,1,NaN,NaN,NaN,NaN
22,Amazon,2020,1,11.821827,35.959275,16.183049,22.200044
25,Amazon,2020,1,NaN,NaN,NaN,NaN
1,Amazon,2020,2,4.846317,NaN,NaN,NaN
19,Amazon,2020,2,7.300795,38.924731,10.549588,19.728347
20,Amazon,2020,2,NaN,NaN,NaN,NaN
26,Amazon,2020,2,12.531205,20.356877,17.876519,17.566834
30,Amazon,2020,2,NaN,NaN,NaN,NaN


In [44]:
df["Revenue_TTM"] = (
    df.groupby("Company")["Revenue"]
    .rolling(window=4, min_periods=4)
    .sum()
    .reset_index(level=0, drop=True)
)

df["Net_Income_TTM"] = (
    df.groupby("Company")["Net_Income"]
    .rolling(window=4, min_periods=4)
    .sum()
    .reset_index(level=0, drop=True)
)

df["Operating_Cash_Flow_TTM"] = (
    df.groupby("Company")["Operating_Cash_Flow"]
    .rolling(window=4, min_periods=4)
    .sum()
    .reset_index(level=0, drop=True)
)

df[
    [
        "Company",
        "Year",
        "Quarter",
        "Revenue_TTM",
        "Net_Income_TTM",
        "Operating_Cash_Flow_TTM"
    ]
].head(20)

,Company,Year,Quarter,Revenue_TTM,Net_Income_TTM,Operating_Cash_Flow_TTM
0,Amazon,2020,1,NaN,NaN,NaN
17,Amazon,2020,1,NaN,NaN,NaN
18,Amazon,2020,1,NaN,NaN,NaN
22,Amazon,2020,1,NaN,NaN,NaN
25,Amazon,2020,1,NaN,NaN,NaN
1,Amazon,2020,2,NaN,NaN,NaN
19,Amazon,2020,2,NaN,NaN,NaN
20,Amazon,2020,2,NaN,NaN,NaN
26,Amazon,2020,2,NaN,NaN,NaN
30,Amazon,2020,2,NaN,NaN,NaN


In [46]:
quarterly_cols = [
    "Company",
    "cik",
    "form",
    "filed",
    "ddate",
    "Year",
    "Quarter",
    "Revenue",
    "Gross_Profit",
    "Operating_Income",
    "Net_Income",
    "Operating_Cash_Flow",
    "Equity",
    "Profit_Margin",
    "Gross_Margin",
    "Operating_Margin",
    "ROE",
    "Cash_Flow_Margin",
    "Sector",
    "ROE_Adjusted",
    "ROE_Flag",
    "OCF_Category",
    "Cash_Flow_Margin_Adj",
    "JPMorgan_OCF_Note",
    "Has_Financial_Data",
    "Data_Period_Note",
    "Revenue_YoY_Growth",
    "Revenue_TTM",
    "Net_Income_TTM",
    "Operating_Cash_Flow_TTM",
    "Profit_Margin_Clean",
    "FHS_PM_Score",
    "FHS_CFM_Score",
    "ROE_for_FHS",
    "FHS_V2"
]

quarterly_export = df[quarterly_cols].copy()

quarterly_export.to_csv(
    "SEC_Dashboard_CORRECTED.csv",
    index=False
)

print("Exported SEC_Dashboard_CORRECTED.csv")
print("Shape:", quarterly_export.shape)

quarterly_export.head()

Exported SEC_Dashboard_CORRECTED.csv
Shape: (398, 35)


,Company,cik,form,filed,ddate,Year,Quarter,Revenue,Gross_Profit,Operating_Income,Net_Income,Operating_Cash_Flow,Equity,Profit_Margin,Gross_Margin,Operating_Margin,ROE,Cash_Flow_Margin,Sector,ROE_Adjusted,ROE_Flag,OCF_Category,Cash_Flow_Margin_Adj,JPMorgan_OCF_Note,Has_Financial_Data,Data_Period_Note,Revenue_YoY_Growth,Revenue_TTM,Net_Income_TTM,Operating_Cash_Flow_TTM,Profit_Margin_Clean,FHS_PM_Score,FHS_CFM_Score,ROE_for_FHS,FHS_V2
0,Amazon,1018724,10-K,2021-02-03,2020-03-31,2020,1,7.545200e+10,NaN,3.989000e+09,2.535000e+09,NaN,NaN,3.359752,NaN,5.286805,NaN,NaN,Technology,NaN,,Operating,NaN,,1,Q1 2020 - Financial Data Available,NaN,NaN,NaN,NaN,3.359752,2.662193,NaN,NaN,NaN
17,Amazon,1018724,10-Q,2020-05-01,2020-03-31,2020,1,7.545200e+10,NaN,3.989000e+09,1.056300e+10,3.064000e+09,6.527200e+10,13.999629,NaN,5.286805,16.183049,4.060860,Technology,16.183049,,Operating,4.060860,,1,Q1 2020 - Financial Data Available,0.0,NaN,NaN,NaN,13.999629,11.821827,6.838617,16.183049,12.007813
18,Amazon,1018724,10-Q,2020-07-31,2020-03-31,2020,1,NaN,NaN,NaN,NaN,NaN,6.527200e+10,NaN,NaN,NaN,NaN,NaN,Technology,NaN,,Operating,NaN,,0,Q1 2020 - Balance Sheet Only,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22,Amazon,1018724,10-Q,2021-04-30,2020-03-31,2020,1,7.545200e+10,NaN,3.989000e+09,1.056300e+10,3.973200e+10,6.527200e+10,13.999629,NaN,5.286805,16.183049,52.658644,Technology,16.183049,,Operating,52.658644,,1,Q1 2020 - Financial Data Available,0.0,NaN,NaN,NaN,13.999629,11.821827,35.959275,16.183049,22.200044
25,Amazon,1018724,10-Q,2021-07-30,2020-03-31,2020,1,NaN,NaN,NaN,NaN,NaN,6.527200e+10,NaN,NaN,NaN,NaN,NaN,Technology,NaN,,Operating,NaN,,0,Q1 2020 - Balance Sheet Only,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [48]:
usable = df[df["Has_Financial_Data"] == 1].copy()

annual = usable.groupby(
    [
        "Company",
        "Year",
        "Sector"
    ],
    as_index=False
).agg(
    Quarters_Available=("Quarter", "nunique"),
    Revenue_Annual=("Revenue", "sum"),
    Net_Income_Annual=("Net_Income", "sum"),
    Operating_Cash_Flow_Annual=("Operating_Cash_Flow", "sum"),
    Gross_Profit_Annual=("Gross_Profit", "sum"),
    Operating_Income_Annual=("Operating_Income", "sum"),
    Equity_EOY=("Equity", "last")
)

annual["Data_Complete"] = np.where(
    annual["Quarters_Available"] >= 3,
    "Complete",
    "Partial"
)

annual["Profit_Margin_Annual"] = safe_divide(
    annual["Net_Income_Annual"],
    annual["Revenue_Annual"]
) * 100

annual["OCF_Margin_Annual"] = safe_divide(
    annual["Operating_Cash_Flow_Annual"],
    annual["Revenue_Annual"]
) * 100

annual["Operating_Margin_Annual"] = safe_divide(
    annual["Operating_Income_Annual"],
    annual["Revenue_Annual"]
) * 100

annual["ROE_Annual"] = safe_divide(
    annual["Net_Income_Annual"],
    annual["Equity_EOY"]
) * 100

annual["Gross_Margin_Annual"] = safe_divide(
    annual["Gross_Profit_Annual"],
    annual["Revenue_Annual"]
) * 100

annual["OCF_Category"] = np.where(
    annual["Company"] == "JPMorgan Chase",
    "Banking OCF",
    "Operating Cash Flow"
)

annual["OCF_Comparable"] = np.where(
    annual["Company"] == "JPMorgan Chase",
    "No",
    "Yes"
)

annual = annual.sort_values(
    [
        "Company",
        "Year"
    ]
).copy()

annual["Revenue_YoY_Pct"] = (
    annual.groupby("Company")["Revenue_Annual"]
    .pct_change()
    * 100
)

annual.head()

,Company,Year,Sector,Quarters_Available,Revenue_Annual,Net_Income_Annual,Operating_Cash_Flow_Annual,Gross_Profit_Annual,Operating_Income_Annual,Equity_EOY,Data_Complete,Profit_Margin_Annual,OCF_Margin_Annual,Operating_Margin_Annual,ROE_Annual,Gross_Margin_Annual,OCF_Category,OCF_Comparable,Revenue_YoY_Pct
0,Amazon,2020,Technology,4,2.007938e+12,1.187390e+11,4.264610e+11,0.0,1.167750e+11,9.340400e+10,Complete,5.913479,21.238753,5.815668,127.124106,0.0,Operating Cash Flow,Yes,NaN
1,Amazon,2021,Technology,4,2.404402e+12,1.973790e+11,3.917860e+11,0.0,1.352050e+11,1.382450e+11,Complete,8.209068,16.294530,5.623228,142.774784,0.0,Operating Cash Flow,Yes,19.744833
2,Amazon,2022,Technology,4,2.746863e+12,-2.388200e+10,2.036750e+11,0.0,6.973800e+10,1.460430e+11,Complete,-0.869428,7.414822,2.538823,-16.352718,0.0,Operating Cash Flow,Yes,14.243084
3,Amazon,2023,Technology,4,2.220959e+12,1.090180e+11,3.758130e+11,0.0,1.334450e+11,2.018750e+11,Complete,4.908600,16.921204,6.008440,54.002724,0.0,Operating Cash Flow,Yes,-19.145622
4,Amazon,2024,Technology,4,1.538293e+12,1.585980e+11,4.217010e+11,0.0,2.239660e+11,2.859700e+11,Complete,10.309999,27.413568,14.559385,55.459664,0.0,Operating Cash Flow,Yes,-30.737443


In [50]:
annual["FHS_PM_Score"] = (
    annual.groupby("Year")["Profit_Margin_Annual"]
    .transform(minmax_score)
)

annual["FHS_ROE_Score"] = (
    annual.groupby("Year")["ROE_Annual"]
    .transform(minmax_score)
)

annual["OCF_Margin_For_Score"] = np.where(
    annual["Company"] == "JPMorgan Chase",
    np.nan,
    annual["OCF_Margin_Annual"]
)

annual["FHS_OCF_Score"] = (
    annual.groupby("Year")["OCF_Margin_For_Score"]
    .transform(minmax_score)
)

annual["Financial_Health_Score"] = (
    annual["FHS_PM_Score"] * 0.35 +
    annual["FHS_ROE_Score"] * 0.30 +
    annual["FHS_OCF_Score"].fillna(50) * 0.35
)

annual[
    [
        "Company",
        "Year",
        "Profit_Margin_Annual",
        "ROE_Annual",
        "OCF_Margin_Annual",
        "Financial_Health_Score"
    ]
].head(20)

,Company,Year,Profit_Margin_Annual,ROE_Annual,OCF_Margin_Annual,Financial_Health_Score
0,Amazon,2020,5.913479,127.124106,21.238753,17.095929
1,Amazon,2021,8.209068,142.774784,16.294530,2.941196
2,Amazon,2022,-0.869428,-16.352718,7.414822,0.000000
3,Amazon,2023,4.908600,54.002724,16.921204,2.376480
4,Amazon,2024,10.309999,55.459664,27.413568,15.522736
5,Amazon,2025,15.973028,28.486518,15.483802,12.774409
6,Apple,2020,20.897046,499.417130,33.323034,74.208139
7,Apple,2021,26.261514,730.117333,34.963315,65.826510
8,Apple,2022,23.035019,912.385636,34.749312,80.234463
9,Apple,2023,30.480620,716.673414,41.006192,89.008771


In [52]:
annual_cols = [
    "Company",
    "Year",
    "Sector",
    "Quarters_Available",
    "Data_Complete",
    "Revenue_Annual",
    "Net_Income_Annual",
    "Operating_Cash_Flow_Annual",
    "Gross_Profit_Annual",
    "Operating_Income_Annual",
    "Equity_EOY",
    "Profit_Margin_Annual",
    "OCF_Margin_Annual",
    "Operating_Margin_Annual",
    "ROE_Annual",
    "OCF_Category",
    "OCF_Comparable",
    "Gross_Margin_Annual",
    "Revenue_YoY_Pct",
    "Financial_Health_Score"
]

annual_export = annual[annual_cols].copy()

annual_export.to_csv(
    "SEC_Annual_Summary.csv",
    index=False
)

print("Exported SEC_Annual_Summary.csv")
print("Shape:", annual_export.shape)

annual_export.head()

Exported SEC_Annual_Summary.csv
Shape: (30, 20)


,Company,Year,Sector,Quarters_Available,Data_Complete,Revenue_Annual,Net_Income_Annual,Operating_Cash_Flow_Annual,Gross_Profit_Annual,Operating_Income_Annual,Equity_EOY,Profit_Margin_Annual,OCF_Margin_Annual,Operating_Margin_Annual,ROE_Annual,OCF_Category,OCF_Comparable,Gross_Margin_Annual,Revenue_YoY_Pct,Financial_Health_Score
0,Amazon,2020,Technology,4,Complete,2.007938e+12,1.187390e+11,4.264610e+11,0.0,1.167750e+11,9.340400e+10,5.913479,21.238753,5.815668,127.124106,Operating Cash Flow,Yes,0.0,NaN,17.095929
1,Amazon,2021,Technology,4,Complete,2.404402e+12,1.973790e+11,3.917860e+11,0.0,1.352050e+11,1.382450e+11,8.209068,16.294530,5.623228,142.774784,Operating Cash Flow,Yes,0.0,19.744833,2.941196
2,Amazon,2022,Technology,4,Complete,2.746863e+12,-2.388200e+10,2.036750e+11,0.0,6.973800e+10,1.460430e+11,-0.869428,7.414822,2.538823,-16.352718,Operating Cash Flow,Yes,0.0,14.243084,0.000000
3,Amazon,2023,Technology,4,Complete,2.220959e+12,1.090180e+11,3.758130e+11,0.0,1.334450e+11,2.018750e+11,4.908600,16.921204,6.008440,54.002724,Operating Cash Flow,Yes,0.0,-19.145622,2.376480
4,Amazon,2024,Technology,4,Complete,1.538293e+12,1.585980e+11,4.217010e+11,0.0,2.239660e+11,2.859700e+11,10.309999,27.413568,14.559385,55.459664,Operating Cash Flow,Yes,0.0,-30.737443,15.522736


In [54]:
print("Quarterly file shape:", quarterly_export.shape)
print("Annual file shape:", annual_export.shape)

print("\nQuarterly companies:")
print(quarterly_export["Company"].value_counts())

print("\nAnnual companies:")
print(annual_export["Company"].value_counts())

print("\nHas financial data count:")
print(quarterly_export["Has_Financial_Data"].value_counts())

print("\nAnnual data completeness:")
print(annual_export["Data_Complete"].value_counts())

print("\nQuarterly columns:")
print(quarterly_export.columns.tolist())

print("\nAnnual columns:")
print(annual_export.columns.tolist())

Quarterly file shape: (398, 35)
Annual file shape: (30, 20)

Quarterly companies:
Company
Amazon            99
Apple             98
Tesla             78
Microsoft         63
JPMorgan Chase    60
Name: count, dtype: int64

Annual companies:
Company
Amazon            6
Apple             6
JPMorgan Chase    6
Microsoft         6
Tesla             6
Name: count, dtype: int64

Has financial data count:
Has_Financial_Data
1    239
0    159
Name: count, dtype: int64

Annual data completeness:
Data_Complete
Complete    30
Name: count, dtype: int64

Quarterly columns:
['Company', 'cik', 'form', 'filed', 'ddate', 'Year', 'Quarter', 'Revenue', 'Gross_Profit', 'Operating_Income', 'Net_Income', 'Operating_Cash_Flow', 'Equity', 'Profit_Margin', 'Gross_Margin', 'Operating_Margin', 'ROE', 'Cash_Flow_Margin', 'Sector', 'ROE_Adjusted', 'ROE_Flag', 'OCF_Category', 'Cash_Flow_Margin_Adj', 'JPMorgan_OCF_Note', 'Has_Financial_Data', 'Data_Period_Note', 'Revenue_YoY_Growth', 'Revenue_TTM', 'Net_Income_TTM', 

In [56]:
quarterly_export[
    [
        "Company",
        "Year",
        "Quarter",
        "ROE",
        "ROE_Adjusted",
        "ROE_Flag",
        "Cash_Flow_Margin",
        "Cash_Flow_Margin_Adj",
        "Has_Financial_Data"
    ]
].sample(10, random_state=42)

,Company,Year,Quarter,ROE,ROE_Adjusted,ROE_Flag,Cash_Flow_Margin,Cash_Flow_Margin_Adj,Has_Financial_Data
212,JPMorgan Chase,2020,1,1.096600,1.096600,,-424.552782,NaN,1
393,Tesla,2025,2,1.515896,1.515896,,20.874822,20.874822,1
5,Amazon,2021,4,24.133965,24.133965,,9.860543,9.860543,1
219,JPMorgan Chase,2020,4,NaN,NaN,,NaN,NaN,0
96,Amazon,2024,4,NaN,NaN,,NaN,NaN,0
76,Amazon,2024,2,10.114740,10.114740,,72.951878,72.951878,1
383,Tesla,2023,3,NaN,NaN,,38.055675,38.055675,1
86,Amazon,2025,1,21.559698,21.559698,,10.930383,10.930383,1
230,JPMorgan Chase,2022,1,2.896827,2.896827,,-136.461894,NaN,1
145,Apple,2021,2,115.322029,28.830507,TTM_Anomaly_Adjusted,29.681686,29.681686,1
